`Note For Kamila and Hamza : The current implementation found below was first developed on the training data for the unkown NARX model. The methodology found below will be exactly the same once we will merge and I will be able to obtain the training data from the two pilots simulations but I wanted to be sure how to implement everything before`

In [1]:
import numpy as np 
import pandas as pd

# 2.C - The Baseline Approach

## 2.C.1 - Obtaining the outputs and features vectors 

### Loading the data 

The first step in the baseline approach is to load the data from the .npz file and store it properly. In this step, we aim to obtain : 

$$
u_1^{tr}(k),\ u_2^{tr}(k),\ y_1^{tr}(k),\ y_2^{tr}(k) \quad k=1, \dots, N_{tr},
$$
where $N_{tr}$ corresponds to the size of the training set ($N_{tr} = 1000$ in the case of the unkown NARX model). Let us therefore extract these variables.

In [2]:
#Load the data from the npz file
data = np.load('C:/Users/guill/Desktop/project-2025-2026-team-10-1/StudentdataNARX.npz')

# Extarct the tarining data from the two corresponding time series in the datafile
U_train = data["Utr"]
Y_train = data["Ytr"]

# Print the shapes of teh training data
U_train.shape, Y_train.shape

((1000, 2), (1000, 2))

At this point, the two arrays defined above, $U_{train}$ and $Y_{train}$ respectively contains values for the inputs in the following way :

$$
U_{train}[i] = ((u_1(i+1)),\ u_2(i+1)) \quad i = 0, \dots, 999
$$

and 

$$
Y_{train}[i] = ((y_1(i+1)),\ y_2(i+1)) \quad i = 0, \dots, 999.
$$

Indeed, for example, $U_{train}[0] = \left( u_1(1),\ u_2(1) \right)$, while $U_{train}[999] = \left( u_1(1000) ,\ u_2(1000) \right)$. Therefore, we don't have access to values for $k=0$ or $k < 0$ yet, we will have to create them ourselves. 

### Adding the Initial Conditions

The next step is therefore to add the initial conditions to these two arrays, i.e. given the project requirements we want to consider that : 

$$
u_1(0) = 0, \quad u_2(0) = 0, \quad y_1(0) = 0, \quad y_2(0) = 0.
$$

This is easily done using the numpy package. 

In [3]:
# Definition of the initial conditiions 
U_init = np.zeros((1,2))
Y_init = np.zeros((1,2))

# The initial conditions are added tpo the training data previously loaded
U_full = np.vstack([U_init, U_train])
Y_full = np.vstack([Y_init, Y_train])

U_full.shape, Y_full.shape

((1001, 2), (1001, 2))

`Note` : Now that the initial conditions have been added, the new arrays index and the inputs/outputs values correspond. Indeed, we now ahve : 

$$
U_{full}[k] = \left(u_1(k), u_2(k) \right) \quad k = 0, 1,\dots, 1000, 
$$

and 

$$
Y_{full}[k] = \left(y_1(k), y_2(k) \right) \quad k = 0, 1,\dots, 1000, 
$$

which will be much nicer to deal with in what follows. 

### Creating the original dataframe

Following the extraction steps from above and the addition of teh initial conditions to the two outputs/inputs arrays, we are now going to consolidate everyhting into a `pandas` dataframe with the following structure : 

<div align="center">

| $k$ | $u_1(k)$ | $u_2(k)$ | $y_1(k)$ | $y_2(k)$ |
|:---:|:---:|:---:|:---:|:---:|
| $0$ | ... | ... | ... | ... | 
| $1$ | ... | ... | ... | ... | 
| $\vdots$ | $\vdots$ | $\vdots$ | $\vdots$ | $\vdots$ | 
| $1000$ | ... | ... | ... | ... | 

</div>

Creating such a dataframe is an easy task given what we discussed above and the use of pandas.

In [4]:
# Creation of a dataframe combining both the inputs from the arrays containing the training data
# and the initial conditions 
df = pd.DataFrame({
    'u1(k)': U_full[:, 0],
    'u2(k)': U_full[:, 1],
    'y1(k)': Y_full[:, 0],
    'y2(k)': Y_full[:, 1],
})

df

,u1(k),u2(k),y1(k),y2(k)
0,0.000000,0.000000,0.000000,0.000000
1,1.225856,0.043213,0.000000,0.000000
2,-0.379083,-0.411898,0.042170,0.720120
3,0.683140,-0.921644,-0.016521,-0.222306
4,0.374853,-0.572683,-0.235810,0.020866
...,...,...,...,...
996,0.346771,0.532575,0.054010,-0.811267
997,-0.832074,-0.479654,0.076635,-0.082176
998,0.272503,0.511913,-0.163845,-0.378072
999,0.509398,0.395705,0.017180,-0.398397


Now, we can easily determine if there is any kissing values in this dataframe

In [5]:
# Check for missing values in the dataframe
missing_values = df.isnull().sum()
print(missing_values)

u1(k)    0
u2(k)    0
y1(k)    0
y2(k)    0
dtype: int64


Not having found any, we can then extract basic statistics aboput the data

In [6]:
# description of the dataframe to get an overviwes of the data
df.describe()

,u1(k),u2(k),y1(k),y2(k)
count,1001.000000,1001.000000,1001.000000,1001.000000
mean,-0.026783,-0.034280,-0.017703,-0.376505
std,0.613926,0.611268,0.220486,0.579543
min,-2.369137,-2.220139,-2.072976,-4.856286
25%,-0.448497,-0.465670,-0.057469,-0.618913
50%,-0.031056,-0.040078,-0.001225,-0.246846
75%,0.374853,0.378868,0.047675,-0.006348
max,1.791663,1.613999,1.950458,1.071298


### Defining the Outputs and features vectors

To prepare the data from the aboce datframe for training, for a given row $k$, we first need to extract the target vector $y(k)$ and the features vector $X(k)$ based on the optimal parameters $n_a, n_b$ and $d$ found before. 

#### 1. The target vector $y(k)$ 

The target vector represents our ground-truth values for the next time step: 

$$
y[k] = \left( y_1(k+1), \ y_2(k+1) \right)
$$

This is is easily obatined from the original dataframe by shifting the original outputs columns by one : 

In [7]:
# Shifting the ouputs columns to creqte the target varaibales for the next time step (y1(k+1), y2(k+1))
y = df[['y1(k)', 'y2(k)']].shift(-1).rename(columns={'y1(k)': 'y1(k+1)','y2(k)': 'y2(k+1)'})
y

,y1(k+1),y2(k+1)
0,0.000000,0.000000
1,0.042170,0.720120
2,-0.016521,-0.222306
3,-0.235810,0.020866
4,-0.080339,-0.352666
...,...,...
996,0.076635,-0.082176
997,-0.163845,-0.378072
998,0.017180,-0.398397
999,0.051686,0.009760


However, as can be seen in the two columns dataframe above, the last row is full of NaNs. Indeed, we started from a dataframe wehere roow $k$ contained $y1(k)$ and $y2(k)$ for $k=0, 1, \dots, 1000$. Therefore, shifting everything by one, the row of index $k=1000$ now contains the values of $y1(1001)$ and $y2(1001)$, which are obviously NaN given that they are unknown from the training data. To solve this issue, we are simply going to discard the last row of the new dataframe given it has no use for training. We threfore end up with a dataframe of size $(1000,2)$ for the outputs.

In [8]:
# Drop the last row of the outputs dataframe given it has NaN inside of it
y = y.iloc[:-1].reset_index(drop=True)
y

,y1(k+1),y2(k+1)
0,0.000000,0.000000
1,0.042170,0.720120
2,-0.016521,-0.222306
3,-0.235810,0.020866
4,-0.080339,-0.352666
...,...,...
995,0.054010,-0.811267
996,0.076635,-0.082176
997,-0.163845,-0.378072
998,0.017180,-0.398397


#### 2. The feature vector/matrix ($X$)

The features vector $X(k)$ is constructed from the lagged observations of both inputs and ouputs. In `pandas`, these are generated by properly shifting the original columns down (e.g., $y_1(k-4)$ is the values of  $y_1(k-4)$ from $4$ rows below). Our goal is to obtain a feqture vector defined qs : 

$$
X[k] = \left( y_1(k-d), \dots, y_1(k-d-n_a), \ y_2(k-d), \dots, y_2(k-d-n_a), \ u_1(k), \dots, u_1(k-n_b), \ u_2(k), \dots, u_2(k-n_b) \right).
$$

However, beforte obtqining the feqture mqtrix and vectors by shifting the original dataframe accordingly, given that we dsicarded the last row of the ouputs daframe (the row of index $1000$), the same has to be done with the last row of the data. Indeed, for examples, the values of $u_1(1000)$ and $u_2(1000)$ will never be used to compute the next predicitons. 

In [9]:
# Drop tyhe last row which has Nans in the corresponding target columns
data = df.iloc[:-1].reset_index(drop=True)
data

,u1(k),u2(k),y1(k),y2(k)
0,0.000000,0.000000,0.000000,0.000000
1,1.225856,0.043213,0.000000,0.000000
2,-0.379083,-0.411898,0.042170,0.720120
3,0.683140,-0.921644,-0.016521,-0.222306
4,0.374853,-0.572683,-0.235810,0.020866
...,...,...,...,...
995,-0.286226,0.104715,-0.084960,-0.198725
996,0.346771,0.532575,0.054010,-0.811267
997,-0.832074,-0.479654,0.076635,-0.082176
998,0.272503,0.511913,-0.163845,-0.378072


Now, to obtain the features dataframe from the above dataframe, we ned to properly shift everyhting following the best $n_a, n_b$ and $d$ found before in order to have every releavnt features fro predidciting $y_1(k+1)$ and $y_2(k+1)$ on row $k$. To do this, let us create an function `create_lagged_features` that takes the above dataframe and the parmatersd as arguments, and returns a dataframe with all the releavant features to compute $y_1(k+1)$ and $y_2(k+1)$ on row $k$.

`Note` : Shifting the dataframe only following this method would produce starting NaNs values in the first few rows of the returned daframe. Indeed for example if $d = 5$, on row $k = 3$ we would end up with $y_1(k-d) = y_1(3-5) = y_1(-2)$, which is obviously a NaN. To paliate this issue and following project requirements, we are going to fill all these starting NaNs values in the dataframe with zeros.

In [10]:
def create_lagged_features(df: pd.DataFrame, n_a: int, n_b: int, d: int) -> pd.DataFrame:

    # Create a copy of the original dataframme
    dataframe = df.copy()

    # Initialize the lags for y1 and y2 from scratch
    y1_lags = []
    y2_lags = []

    # For the outputs y1 and y2, we need to loop from y(k-d) to y(k-d-n_a)
    for i in range(d, d + n_a + 1):

        y1_col, y2_col = f'y1(k-{i})', f'y2(k-{i})'

        # Shift the datframe accoridngly from the origi,ql columns
        dataframe[y1_col] = dataframe['y1(k)'].shift(i)
        dataframe[y2_col] = dataframe['y2(k)'].shift(i)

        # Append the results to the lags 
        y1_lags.append(y1_col)
        y2_lags.append(y2_col)

    # For the inputs, we want to keep the current values 
    u1_lags = ['u1(k)']
    u2_lags = ['u2(k)']

    # Here, we need to loop from u(k-1) to u(k-n_b)
    for i in range(1, n_b + 1):

        u1_col, u2_col = f'u1(k-{i})', f'u2(k-{i})'

        dataframe[u1_col] = dataframe['u1(k)'].shift(i)
        dataframe[u2_col] = dataframe['u2(k)'].shift(i)

        # Append the results to the lags
        u1_lags.append(u1_col)
        u2_lags.append(u2_col)

    # Create a proper order for the columns in the retruned dataframe
    columns_order = y1_lags + y2_lags + u1_lags + u2_lags

    # As explained in the note from the above cell, we fill all the starting Nans values with zeros
    dataframe[columns_order] = dataframe[columns_order].fillna(0.0)

    result = dataframe[columns_order]

    return result

In [11]:
# Test of the methodology
n_a = 6
n_b = 5
d = 4

# The feqtures datafrme is simply obtained by calling the function with the datafrme and the parmaters values.
X = create_lagged_features(data, n_a, n_b, d)
X.head(10)

,y1(k-4),y1(k-5),y1(k-6),y1(k-7),y1(k-8),y1(k-9),y1(k-10),y2(k-4),y2(k-5),y2(k-6),...,u1(k-2),u1(k-3),u1(k-4),u1(k-5),u2(k),u2(k-1),u2(k-2),u2(k-3),u2(k-4),u2(k-5)
0,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.043213,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,-0.411898,0.043213,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,1.225856,0.000000,0.000000,0.000000,-0.921644,-0.411898,0.043213,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,-0.379083,1.225856,0.000000,0.000000,-0.572683,-0.921644,-0.411898,0.043213,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.683140,-0.379083,1.225856,0.000000,-0.760764,-0.572683,-0.921644,-0.411898,0.043213,0.000000
6,0.042170,0.000000,0.000000,0.00000,0.0,0.0,0.0,0.720120,0.000000,0.000000,...,0.374853,0.683140,-0.379083,1.225856,0.052034,-0.760764,-0.572683,-0.921644,-0.411898,0.043213
7,-0.016521,0.042170,0.000000,0.00000,0.0,0.0,0.0,-0.222306,0.720120,0.000000,...,0.256665,0.374853,0.683140,-0.379083,0.057735,0.052034,-0.760764,-0.572683,-0.921644,-0.411898
8,-0.235810,-0.016521,0.042170,0.00000,0.0,0.0,0.0,0.020866,-0.222306,0.720120,...,0.630456,0.256665,0.374853,0.683140,-0.177296,0.057735,0.052034,-0.760764,-0.572683,-0.921644
9,-0.080339,-0.235810,-0.016521,0.04217,0.0,0.0,0.0,-0.352666,0.020866,-0.222306,...,-0.048792,0.630456,0.256665,0.374853,-0.322927,-0.177296,0.057735,0.052034,-0.760764,-0.572683


From there, the current situation is tghat on row $k$ ($ k = 0, 1, \dots, 999$), we respectively have, for the outputs y : 

<div align="center">

| $k$ | $y_1(k+1)$ | $y_2(k+1)$ |
|:---:|:---:|:---:|

</div>

and for the features matrix 

<div align="center">

| $k$ | $y_1(k-d)$ | $\dots$ | $y_1(k-d-n_a)$ | $y_2(k-d)$ | $\dots$ | $y_2(k-d-n_a)$ | $u_1(k)$ | $\dots$ | $u_1(k-n_b)$ |$u_2(k)$ | $\dots$ | $u_2(k-n_b)$ |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|

</div>

Therefore, we have a perfect match between the two dataframes in terms of rows with the feature vector $X[k]$ and the ground-truth ouputs $y[k]$. This is exactly what is needed to train our models, and we can tehrefore pass to the next step. 




## 2.C.2 Training non-deep learning models. 

Our objective when training models is to identify the optimal mapping functions that minimize speicif evqluations metrics, such as NMSE or RMSE. 

Concretely, we define the feature matrix/vector $X(k)$ and we aim to estimate functions $\hat{f}_1$ and $\hat{f}_1$ such that :

$$
\hat{y}_1(k+1) = \hat{f}_1\left( X(k) \right), \quad \hat{y}_2(k+1) = \hat{f}_2\left( X(k) \right)
$$

The goal is to ensure that these predicitons are as close as possible to the ground-truth values $y_1(k+1)$ and $y_2(k+1)$ from the training data, based on our cvhosen metrics. 

### Utility functions for Model Evaluation 

Before training our candidate models, we define the necessary tools used for a rigourous evaluation. To align with the project objectives, we must look beyond simple One-Step Ahead (OSA in what follows) predicitons and asses how our model behaves over long horizons. 

To this end, let us first implement a `compute_metrics` function to compute the **Root Mean Squared Error (RMSE)** and the **Normalized Mean Squared Error (NMSE)**, which are defined as 

$$
\textrm{RMSE} = \sqrt{\frac{1}{2N_{ts}}\sum_{k=1}^{N_{ts}}\sum_{j=1}^2 \left( y_{j}^{ts}(k) - \hat{y}_{j}(k)\right)^2}
$$

and

$$
\textrm{NMSE} = \frac{\sum_{k=1}^{N_{ts}}\sum_{j=1}^2 \left( y_{j}^{ts}(k) - \hat{y}_{j}(k)\right)^2}{\sum_{k=1}^{N_{ts}}\sum_{j=1}^2 \left(y_{j}^{ts}(k) - \bar{y}_{j}\right)^2},
$$

where 
* $N_{ts}$ is the total number of time steps in the validation/test sequence.
* $y_{j}^{ts}(k)$ represents the ground-truth value for the $j$-th output at time $k$.
* $\hat{y}_{j}(k)$ represents the model's prediction for the $j$-th output at time $k$.
* $\bar{y}_{j}$ is the mean value of the $j$-th ground-truth output over the sequence.

In [12]:
def compute_metrics(y_true, y_pred):
    """
    Takes the groun-truth values for the traget and the asocciated prediciton as arguments

    Compute the RMSE and NMSE for these values. 
    """

    # Convert them to arrays if not already
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # The numerator is common to both of the expressions. 
    numerator = np.sum((y_true - y_pred) ** 2)

    # Ciompute RMSE
    rmse_denom = 2 * len(y_true)
    rmse = np.sqrt(numerator/rmse_denom)

    # Compute NMSE
    means = np.mean(y_true, axis=0)
    nmse_denom = np.sum((y_true - means) ** 2)
    nmse = numerator/nmse_denom

    return rmse, nmse

Then, we note that when we are going to train our sklearn models using the `fit` functiuon and then obtain predictions using `predict`, this will be done in a One-Step-Ahead way : The p^redicitons are based on the previous ground-truth values. 

However, givne the context of the project, this is not what we want. Indeed, what we need is the model to make the next predicition by feeding its own past predicitons back into the model. In the context of the test set, we won't any ground-truth reference value like we would with the training data.

Lret us tehrefore define a `multiple_steps_ahead` function to feed back to the model its own previous preicitons.


In [13]:
def multiple_steps_ahead(model, X_val, n_a: int, d:int) -> np.ndarray:

    # Initialize the total number of steps corrsponding to the feature matrix size
    n_steps = len(X_val)

    # Initialize all the predicitons to zeros
    predictions = np.zeros((n_steps, 2))

    # Make a copy of the validation data
    X_sim = X_val.copy()

    # Foudn the indices of the validation set in teh case of validation 
    # Else the start of the dataframe in test mode 
    val_indices = X_val.index

    # Loop over all the predicitons to feed them back to the model
    for k in range(n_steps):
        
        # Get the index in the datafrme
        actual_idx = val_indices[k]

        for lag_val in range(d, d + n_a + 1):

            pred_idx = k - lag_val - 1

            if pred_idx >= 0:

                y1_col = f'y1(k-{lag_val})'
                y2_col = f'y2(k-{lag_val})'

                # If the columns already exist, add the previous prdicitons
                if y1_col in X_sim.columns:
                    X_sim.at[actual_idx, y1_col] = predictions[pred_idx, 0]

                if y2_col in X_sim.columns:  
                    X_sim.at[actual_idx, y2_col] = predictions[pred_idx, 1]

        # Once all the previous predictions have been added/the dataframe was modified
        # We can use tem to predict the next one using the updated row
        current_prediciton = model.predict(X_sim.loc[[actual_idx]])

        # Add the current prediciton to the list
        predictions[k] = current_prediciton[0]

    # Once the loo^p is obver we can return the all the precitions
    return predictions

### Model Training and Cross-Validation

Having found our target and features vectors and with our tuility fucntions defined, we now proceed to the training and evaluation phases.

#### Time-Series Cross Validation 

Our systrem is inherently time dependent. Because of that, standard k-fold cross validation is inappropirate as it would involve prediciting past values using future data. Instead, we are going to use the `TimeSeriesSplit` method from `sklearn.model_selection`. This validation approach ensure that the training set always precedes the validation set, maintaining the casual integrity of the times series and preventing data leakage. 

#### Candidate Models

In what follows, we are going to evaluate three disticnt non deep-learning models to identify the best fit for our system : 

1. **Linear Regression (OLS)** : It will serves as our baseline, assuming a linear relationship between the lagged features and the outputs. 
2. **K-Nearest Neighbors (KNN)** : Predicts outputs based on teh similarity between the current feature vector and past training examples. 
3. **Random Forest** : Use of mutliple decision trees to capture non-linear interactions between features and outputs. 

#### Evaluation Strategy

For each fold of the time series cross-validation, each model will be evaluated by computing the RMSE and NMSE metrics on the validation part of the fold using two methods : 

- One-Step-Ahead (OSA) : As expalined above, predicitons are made using groud-truth lagged values. It will measure the model's immediate predictive accuracy. 
- Multiple-Steps-Ahead (MSA) : Will measure how well the model reacts to its own previous pre-dicitons being used as ground-truth. 

Given qll these considerations, it is now time to finally train our models.

`NOTE : Right now, no scaling is applied to the data, for random forst it should be ok but we might need to apply it for KNN`

In [14]:
# Import the modyules required fro Time series cross validation and model training
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

In [15]:
# Initialize the time series split 
tscv = TimeSeriesSplit(n_splits=5)

# Initialize the three models with random parameters
models = {
    "OLS": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "RF": RandomForestRegressor(n_estimators=100, random_state=42)
}

# Initialize dictionnaries top store the results for NMSE and RMSE in bothg OSA and MSA
osa_results = {name: [] for name in models}
msa_results = {name: [] for name in models}

# The cross-validation loop on the time series
for i, (train_index, val_index) in enumerate(tscv.split(X)):

    # For each fold, we have a training set and a validation set. 
    # The TimeSeriesSplit allow to easily get these indices and obtain the corrsponding features and outputs values.
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # For clairty, we can implement a small print to see exactly what happens inside each fold
    print(f"--- Fold {i+1} --- Training set size : {len(X_train)} --- Validation set size : {len(X_val)} ---")

    # For each fold, we can now train our models and evaluate the RMSE and NMSE for OSA and MSA
    for name, model in models.items():

        # Training the model on the training set of the fold
        model.fit(X_train, y_train)

        # One-Step-Ahead predicitons on the validation set
        osa_pred = model.predict(X_val)
        rmse_osa, nmse_osa = compute_metrics(y_val, osa_pred)
        osa_results[name].append((rmse_osa, nmse_osa))

        # Multiple-Steps-Ahead predicitons on the validation set
        # Note : This is done by using the multiple_steps_ahead functions previoulsy defined
        msa_pred = multiple_steps_ahead(model, X_val, n_a, d)
        rmse_msa, nmse_msa = compute_metrics(y_val, msa_pred)
        msa_results[name].append((rmse_msa, nmse_msa))

        # Print the results fro the current fold and model
        print(f" {name:12} OSA : RMSE={rmse_osa:.4f}, NMSE={nmse_osa:.4f}")
        print(f" {name:12} MSA : RMSE={rmse_msa:.4f}, NMSE={nmse_msa:.4f}")

    print()

--- Fold 1 --- Training set size : 170 --- Validation set size : 166 ---
 OLS          OSA : RMSE=0.4273, NMSE=1.5348
 OLS          MSA : RMSE=0.4076, NMSE=1.3961
 KNN          OSA : RMSE=0.3235, NMSE=0.8799
 KNN          MSA : RMSE=0.3312, NMSE=0.9220
 RF           OSA : RMSE=0.2692, NMSE=0.6091
 RF           MSA : RMSE=0.2713, NMSE=0.6187

--- Fold 2 --- Training set size : 336 --- Validation set size : 166 ---
 OLS          OSA : RMSE=0.3067, NMSE=0.5738
 OLS          MSA : RMSE=0.3047, NMSE=0.5662
 KNN          OSA : RMSE=0.3556, NMSE=0.7711
 KNN          MSA : RMSE=0.3399, NMSE=0.7046
 RF           OSA : RMSE=0.2288, NMSE=0.3192
 RF           MSA : RMSE=0.2282, NMSE=0.3176

--- Fold 3 --- Training set size : 502 --- Validation set size : 166 ---
 OLS          OSA : RMSE=0.3815, NMSE=0.6093
 OLS          MSA : RMSE=0.3770, NMSE=0.5950
 KNN          OSA : RMSE=0.4181, NMSE=0.7318
 KNN          MSA : RMSE=0.3884, NMSE=0.6314
 RF           OSA : RMSE=0.3180, NMSE=0.4234
 RF           

We can also obtain a final summary by averaging the results obtained over all folds

In [16]:
print("--- Final Average performance ---")
for name in models: 

    osa_rmses = [s[0] for s in osa_results[name]]
    msa_rmses = [s[0] for s in msa_results[name]]
    osa_nmses = [s[1] for s in osa_results[name]]
    msa_nmses = [s[1] for s in msa_results[name]]

    # Compute and print the mean and the std for RMSE and NMSE 
    # For each model, and for both OSA and MSA
    print(f"{name:12} OSA: RMSE = {np.mean(osa_rmses):.4f} (+/- {round(np.std(osa_rmses), 3)}), NMSE = {np.mean(osa_nmses):.4f} (+/- {round(np.std(osa_nmses), 3)}) ")
    print(f"{name:12} MSA: RMSE = {np.mean(msa_rmses):.4f} (+/- {round(np.std(msa_rmses), 3)}), NMSE = {np.mean(msa_nmses):.4f} (+/- {round(np.std(msa_nmses), 3)}) ")

--- Final Average performance ---
OLS          OSA: RMSE = 0.3384 (+/- 0.059), NMSE = 0.7068 (+/- 0.424) 
OLS          MSA: RMSE = 0.3334 (+/- 0.053), NMSE = 0.6746 (+/- 0.37) 
KNN          OSA: RMSE = 0.3566 (+/- 0.053), NMSE = 0.6993 (+/- 0.127) 
KNN          MSA: RMSE = 0.3466 (+/- 0.045), NMSE = 0.6668 (+/- 0.143) 
RF           OSA: RMSE = 0.2493 (+/- 0.061), NMSE = 0.3563 (+/- 0.155) 
RF           MSA: RMSE = 0.2488 (+/- 0.06), NMSE = 0.3560 (+/- 0.159) 


### Optimization Startegy 

The previous analysis provided a first overiew of the situiation but, to ensure we are evaluating the full potential of our candidate modelsd, we now perfom a grid search with cross-validation using the class `GridSearchCV` from `sklearn.model_selection`. This process will iterates through a predefined set of hyperparameters, selecting the combiantion taht minimizes both the RMSE and NMSE accross the time series folds.

- For the Random Forest, we are going to tune the number of trees in the forest (`n_estimator`), th emaximum depth of the trees (`max_depth`), and the minimum number of samples required at a leaf node (`min_samples_lead`). This will help balance underfitting and overfitting (variance vs bias).

- For KNN, we optimize the number of neighbors (`n_neighbors`). A smaller $k$ makes the model sensitive to noise, while a larger one can make it too smooth to capture fast dynamic system.

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer 

# Compute an average metric to use as the scorer 
def average_metrics(y_true, y_pred):
    rmse, nmse =  compute_metrics(y_true, y_pred)
    return rmse + nmse / 2 

# This might need to be checked 

# WE want to use a custom scorer based on the average_metrics function
custom_scorer = make_scorer(average_metrics, greater_is_better=False)

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid={
        'n_estimators': [50, 100, 200, 300, 500],
        'max_depth': [None, 5, 10, 20],
        'min_samples_leaf': [1, 2, 5],
    },
    cv=tscv,
    scoring=custom_scorer,
    n_jobs=1,
    verbose=1,
)

rf_grid.fit(X, y)

print(f"Best RF params : {rf_grid.best_params_}")
print(f"Best RF CV Averaged metric : {-rf_grid.best_score_:.4f}")

knn_grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid={'n_neighbors': [3,5,7,10,15,20]},
    cv=tscv,
    scoring=custom_scorer,
    n_jobs=1,
)

knn_grid.fit(X, y)

print(f"Best KNN params : {knn_grid.best_params_}")
print(f"Best KNN CV Averaged metric : {-knn_grid.best_score_:.4f}")

best_rf = rf_grid.best_estimator_
best_knn = knn_grid.best_estimator_

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best RF params : {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 300}
Best RF CV Averaged metric : 0.4134
Best KNN params : {'n_neighbors': 7}
Best KNN CV Averaged metric : 0.6972


`Side note : The above cell takes a LOT of time to run , took almost 5 minutes.`

The **Random Forest** model outperformed KNN by a significant margin. This suggests that the ensemble of decision trees is better equipped to handle the high-dimensional feature space created by our lagged inputs and outputs. The choice of 300 estimators and a minimum of 2 samples per leaf provides a robust balance between model complexity and generalization.


With these optimized estimators, we can now proceed to the final comparison on the full test sequence to assess their long-term stability in a free-run simulation.


### Final Evaluations of Tuned Models

With the optimal hyperparameters identified, we perform a final comparative analysis between the Random Forest and K-Nearest neighbors models. To assess their utility for predictives uses, we evaluate them using both OSA and MSA.


In [ ]:
tuned_models = {"KNN (tuned)": best_knn, "RF (tuned)" : best_rf}

tuned_osa_results = {name: [] for name in tuned_models}
tuned_msa_results = {name: [] for name in tuned_models}

for train_idx, val_idx in tscv.split(X):


    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y.iloc[train_idx], y.iloc[val_idx]

    for name, model in tuned_models.items():

        # One-Step-Ahead predicitons
        model.fit(X_tr, y_tr)
        osa_pred = model.predict(X_vl)
        rmse_osa, nmse_osa = compute_metrics(y_vl, osa_pred)
        tuned_osa_results[name].append((rmse_osa, nmse_osa))

        # Multiple-Step-Ahead predicitons
        model.fit(X_tr, y_tr)
        msa_pred = multiple_steps_ahead(model, X_vl, n_a, d)
        rmse_msa, nmse_msa = compute_metrics(y_vl, msa_pred)
        tuned_msa_results[name].append((rmse_msa, nmse_msa))

print("Tuned model performances")

for name in tuned_models:

    osa_rmses = [s[0] for s in tuned_osa_results[name]]
    osa_nmses = [s[1] for s in tuned_osa_results[name]]
    msa_rmses = [s[0] for s in tuned_msa_results[name]]
    msa_nmses = [s[1] for s in tuned_msa_results[name]]

    print(f"{name:14} OSA: RMSE = {np.mean(osa_rmses):.4f} (+/- {np.std(osa_rmses):.4f}), NMSE = {np.mean(osa_nmses):.4f} (+/- {np.std(osa_nmses):.4f})")
    print(f"{name:14} Sim: RMSE = {np.mean(msa_rmses):.4f} (+/- {np.std(msa_rmses):.4f}), NMSE = {np.mean(msa_nmses):.4f} (+/- {np.std(msa_nmses):.4f})")

--- Tuned model performances
KNN (tuned)    OSA: RMSE = 0.3535 (+/- 0.0514), NMSE = 0.6873 (+/- 0.1242)
KNN (tuned)    Sim: RMSE = 0.3470 (+/- 0.0505), NMSE = 0.6645 (+/- 0.1348)
RF (tuned)     OSA: RMSE = 0.2432 (+/- 0.0653), NMSE = 0.3405 (+/- 0.1590)
RF (tuned)     Sim: RMSE = 0.2425 (+/- 0.0647), NMSE = 0.3388 (+/- 0.1591)


### Conclusion for model selection 

`THIS WILL HAVE TO BE DONE FOR NARX1 AND NARX2 NOT FOR THE UNKOWN NARX`

### Features Importance Analysis

`THIS WILL HAVE TO BE DONE FOR NARX1 AND NARX2 NOT FOR THE UNKOWN NARX`

## 2.C.3 Integration into a scikit-learn pipeline

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
import pandas as pd 
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline 
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

Creation of a custom transformer class following the idea behind the creat_lagged_features function previoulsy defined

Note : Here, we do not need to fill all the NaN values created at the start with zeros. Indeed, this will be easily handled by using SimpleImputer from sklearn.impute to fill them directly.

In [ ]:
class LaggedTransformer(BaseEstimator, TransformerMixin):
    
    """
    Custom transformer for converting a raw NARX DataFrame into lagged features.
    """
   
    def __init__(self, n_a=None, n_b=None, d=None):
        self.n_a = n_a
        self.n_b = n_b
        self.d = d

    def fit(self, X, y):
      
      self.n_a, self.n_b, self.d = 6, 5, 4 # Defaults values

      # self.n_a, self.n_b, self.d = self._find_best_parameters(X,y) -> HAMZA

      return self
   
    def transform(self, data: pd.DataFrame):

        X = data.copy()

        # During multiple-steps-ahead predicitons for the test set, y columns don't exist yet -> Initalized with zeros
        if 'y1(k)' not in X.columns:
            X['y1(k)'] = 0.0
        if 'y2(k)' not in X.columns:
            X['y2(k)'] = 0.0

        # For outputs lags, we need delay from d to d + n_a
        y1_lags, y2_lags = [], []

        for i in range(self.d, self.d + self.n_a + 1):
            y1_col, y2_col = f'y1(k-{i})', f'y2(k-{i})'

            X[y1_col] = X['y1(k)'].shift(i)
            X[y2_col] = X['y2(k)'].shift(i)

            y1_lags.append(y1_col)
            y2_lags.append(y2_col)

        # For inputs lags, we need the current time step k, then k-1 up to k-n_b
        u1_lags, u2_lags = ['u1(k)'], ['u2(k)']

        for i in range(1, self.n_b + 1):
            u1_col, u2_col = f'u1(k-{i})', f'u2(k-{i})'

            X[u1_col] = X['u1(k)'].shift(i)
            X[u2_col] = X['u2(k)'].shift(i)

            u1_lags.append(u1_col)
            u2_lags.append(u2_col)

        # Small modifictaiton to have a proper order 
        final_column_order = y1_lags + y2_lags + u1_lags + u2_lags

        # Side note as expalined before : The starting NaNs values created by the shifts will be delt with using SimpleImputer inside the pipeline.

        return X[final_column_order]
    
    def _find_best_parameters(self, data, y):

        # THIS IS WHERE THE LOGIC IMPLEMENTED BY HAMZA SHOULD GO 
        pass

Now that we have defined our custom transformer class, we need to define a class called ModelSelector. This will consist of we previously did to find the best models given the training data and its best hyperparameters, it will be used inside the pipeline such that when data are flowing htough the pipleine, the custom transfromer class defined above is applied, and then given the transfromed data, the best model and best hyperparameters are found. 

`NOTE : RIGHT NOW THE SCORING IS USING NEG_MEAN_SQUARED_ERROR BUT WE MIGHT WANT TO CHANGE THAT USING A CUSTOM SCORER ?`

In [ ]:
class ModelSelector(BaseEstimator, RegressorMixin):

    """
    Selects the best models among RandomForest, KNN, and LinearRegression by tuning hyperpamraters 
    via GridSearchCV on the already transfromed data obtained via the LaggedTransfromer, then picking the 
    Candidate with the best CV score.
    """

    def fit(self, X, y):

        tscv = TimeSeriesSplit(n_splits=5)
        candidates = []

        print("\nTuning RandomForest...")

        rf_grid = GridSearchCV(
            MultiOutputRegressor(RandomForestRegressor(random_state=42)),
            param_grid = {
                'estimator__n_estimators': [50, 100, 200, 300, 500],
                'estimator__max_depth': [None, 5, 10, 20],
                'estimator__min_samples_leaf': [1, 2, 5]
            },
            cv=tscv,
            scoring='neg_mean_squared_error',
            n_jobs=-1, 
            verbose=0
        )

        rf_grid.fit(X,y)
        candidates.append({
            'name': 'Random Forest', 
            'score': rf_grid.best_score_,
            'model': rf_grid.best_estimator_,
            'params': rf_grid.best_params_
        })

        print("\nTuning KNN...")
        
        knn_grid = GridSearchCV(
            MultiOutputRegressor(KNeighborsRegressor()),
            param_grid={
                'estimator__n_neighbors': [3, 5, 7, 10, 15, 20]
            }, 
            cv=tscv,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        knn_grid.fit(X,y)
        candidates.append({
            'name': 'KNN',
            'score': knn_grid.best_score_,
            'model': knn_grid.best_estimator_,
            'params': knn_grid.best_params_
        })

        print("\nEvaluating Linear Regression...")

        lr = LinearRegression()
        lr_scores = cross_val_score(lr, X, y, cv=tscv, scoring='neg_mean_squared_error')
        lr.fit(X,y)
        candidates.append({
            'name': 'Linear Regression',
            'score': np.mean(lr_scores),
            'model': lr,
            'params': {}
        })

        print("\n=== Model Selection Results ===")
        print(f"{'Model':<25} {'CV MSE':>10} Best Parmas")
        print("-" * 70)

        for c in candidates: 
            params_str = str(c['params']) if c['params'] else "N/A"
            print(f"{c['name']:<25} {-c['score']:>10.4f} {params_str}")

        # Once this is done, we select the best model
        best = max(candidates, key=lambda x: x['score'])
        self.best_model_ = best["model"]

        print(f"\nBest Model: {best['name']}")

        if best['params']:
            print(f"\nBest Hyperparameters: {best['params']}")

        return self
    
    def predict(self, X):
        return self.best_model_.predict(X)

We are now finally ready to integrate verything we have done up to this point inside a pipeline object. This step is in fact now quite easy given the two classes we juste defined. We simply need to use LaggedTransfromer as the transfromer inside the pipeline and ModelSelector as the model. We also implement a scaler using StandardScaler. 

In [23]:
pipeline = Pipeline([
    ('transformer', LaggedTransformer()),
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)), # Fills the starting NaNs values with zeros
    ('scaler', StandardScaler()),
    ('model', ModelSelector())
])

The pipeline is now finally ready to be used ! To test it; let's start from the begining `_THIS WILL NEED TO BE CHANGED LATER BUT THIS IS FOR TEST PRUPOSE`

In [ ]:
data = np.load('C:/Users/guill/Desktop/project-2025-2026-team-10-1/StudentdataNARX.npz')

U_train = data['Utr']
Y_train = data['Ytr']


U_full = np.vstack([np.zeros((1,2)), U_train])
Y_full = np.vstack([np.zeros((1,2)), Y_train])

df = pd.DataFrame({
    'u1(k)': U_full[:, 0],
    'u2(k)': U_full[:, 1],
    'y1(k)': Y_full[:, 0],
    'y2(k)': Y_full[:, 1],
})

y = df[['y1(k)', 'y2(k)']].shift(-1).iloc[:-1].reset_index(drop=True)
data = df.iloc[:-1].reset_index(drop=True)


X_train = data.iloc[:800]
y_train = y.iloc[:800]

X_val_raw = data.iloc[800:]
y_val = y.iloc[800:]

Training Phase

In [ ]:
print("=" * 60)
print("Fitting the pipeline on training data")
print("=" * 60)

pipeline.fit(X_train, y_train)

Fitting the pipeline on training data

[1/3] Tuning RandomForest...

[2/3] Tuning KNN...

[3/3] Evaluating Linear Regression...

=== Model Selection Results ===
Model                         CV MSE Best Parmas
----------------------------------------------------------------------
Random Forest                 0.0598 {'estimator__max_depth': None, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 300}
KNN                           0.1064 {'estimator__n_neighbors': 7}
Linear Regression             0.1130 N/A

Best Model: Random Forest

Best Hyperparameters: {'estimator__max_depth': None, 'estimator__min_samples_leaf': 2, 'estimator__n_estimators': 300}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('transformer', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_a,6
,n_b,5
,d,4
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",0
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True


Validation Phase - With a context window to avoid data leakage on the frist few rows

In [36]:
transformer = pipeline.named_steps['transformer']
context_size = transformer.n_a + transformer.d

X_val_and_context = pd.concat([
    X_train.iloc[-context_size:],
    X_val_raw
]).reset_index(drop=True)

y_pred = pipeline.predict(X_val_and_context)[context_size:]

rmse, nmse = compute_metrics(y_val, y_pred)

print("\n" + "=" * 60)
print("Validation Results")
print("=" * 60)

print(f"RMSE: {rmse:.4f}")
print(f"NMSE:  {nmse:.4f}")


Validation Results
RMSE: 0.1298
NMSE:  0.1138


Right now, the pipelibne seems to be working as intended. However, it currently only predicts prediciton using the One-Step_ahead method. Let us tehrefore implement the multiple step ahead logic as explained before by feeding the model its own rpevious preidicitons. 

In [ ]:
def multiple_steps_ahead(pipeline, U_test, y_init=None):
    """
    Performs multiple-step-ahead prediction from the trained pipeline.
    At each step, predicted outputs are fed back as lag features for the next step. 
    """
    transformer = pipeline.named_steps['transformer']

    # Retrieve the best parameters found during training
    n_a = transformer.n_a
    n_b = transformer.n_b
    d   = transformer.d

    # Number of steps to predict = length of the test set
    n_steps = len(U_test)

    # Context size, i.e how much history we need to fill all lag features
    context = max(n_a + d, n_b) + 1 

    # History buffers initialized with zeros
    y_history = np.zeros((context, 2))
    u_history = np.zeros((context, 2))

    if y_init is not None:
        y_history[-len(y_init):] = y_init

    # Build the expected column order:  must match what the transformer outputs
    # so that the imputer and scaler receive columns in the right order
    y1_lags = [f'y1(k-{i})' for i in range(d, d + n_a + 1)]
    y2_lags = [f'y2(k-{i})' for i in range(d, d + n_a + 1)]
    u1_lags = ['u1(k)'] + [f'u1(k-{i})' for i in range(1, n_b + 1)]
    u2_lags = ['u2(k)'] + [f'u2(k-{i})' for i in range(1, n_b + 1)]
    expected_columns = y1_lags + y2_lags + u1_lags + u2_lags

    predictions = []

    for k in range(n_steps):

        # Current inputs at step k
        u_current = U_test.iloc[k][['u1(k)', 'u2(k)']].values
        u_history = np.vstack([u_history, u_current])

        # Build the feature row manually using historys
        row = {}

        # Output lags: from d to d + n_a
        for i in range(d, d + n_a + 1):
            row[f'y1(k-{i})'] = y_history[-i, 0] if i <= len(y_history) else 0.0
            row[f'y2(k-{i})'] = y_history[-i, 1] if i <= len(y_history) else 0.0

        # Input lags: current (k) then k-1 back to k-n_b
        row['u1(k)'] = u_current[0]
        row['u2(k)'] = u_current[1]
        for i in range(1, n_b + 1):
            row[f'u1(k-{i})'] = u_history[-1-i, 0] if i < len(u_history) else 0.0
            row[f'u2(k-{i})'] = u_history[-1-i, 1] if i < len(u_history) else 0.0

        # Enforce correct column order
        U_row = pd.DataFrame([row])[expected_columns]

        # imputer and scaler
        U_row = pipeline.named_steps['imputer'].transform(U_row)
        U_row = pipeline.named_steps['scaler'].transform(U_row)

        # Predict next outputs using the fitted best model
        y_pred = pipeline.named_steps['model'].predict(U_row)[0] 
        predictions.append(y_pred)

        # Feed prediction back into output history for next step
        y_history = np.vstack([y_history, y_pred])

    return np.array(predictions)

Let's then test this function on the unknown NARX data

In [43]:
data = np.load('C:/Users/guill/Desktop/project-2025-2026-team-10-1/StudentdataNARX.npz')

U_test_1 = data['Uts1']
U_test_2 = data['Uts2']

U_test_1 = pd.DataFrame({
    'u1(k)': U_test_1[:, 0],
    'u2(k)': U_test_1[:, 1],
})

U_test_2 = pd.DataFrame({
    'u1(k)': U_test_2[:, 0],
    'u2(k)': U_test_2[:, 1],
})

y_pred_1 = multiple_steps_ahead(pipeline, U_test_1)
y_pred_2 = multiple_steps_ahead(pipeline, U_test_2)

print(y_pred_1)

[[ 0.0435945   0.61973041]
 [-0.02416648 -0.32829922]
 [-0.20467226  0.00602922]
 ...
 [ 0.02975752 -0.23446058]
 [ 0.07028597  0.1297922 ]
 [ 0.04616018 -0.18488548]]


In [ ]:
y_pred_1.shape # At least the shape is good and the values seem legit with respect to the original dataframe

(1000, 2)

In [ ]:
y_pred_1 = pd.DataFrame({
    'y1(k+1)': y_pred_1[:, 0],
    'y2(k+1)': y_pred_1[:, 1],
})

y_pred_1.describe()

# These stats at least seem to be in the same range as the original one (WHich is good, i guess ?)

,y1(k+1),y2(k+1)
count,1000.000000,1000.000000
mean,-0.016613,-0.372118
std,0.153455,0.500794
min,-1.161844,-2.996664
25%,-0.039806,-0.612774
50%,-0.002626,-0.259711
75%,0.022709,-0.053124
max,1.028541,0.739718


Let's try on the validation set from before in order to compute the rmse and nmse and havena idea

In [46]:
transformer = pipeline.named_steps['transformer']
context_size = transformer.n_a + transformer.d

# --- One-Step-Ahead (existing) ---
X_val_and_context = pd.concat([
    X_train.iloc[-context_size:],
    X_val_raw
]).reset_index(drop=True)

y_pred_osa = pipeline.predict(X_val_and_context)[context_size:]
rmse_osa, nmse_osa = compute_metrics(y_val, y_pred_osa)

# --- Multiple-Step-Ahead ---
# Extract only the input columns from the validation set
U_val = X_val_raw[['u1(k)', 'u2(k)']].reset_index(drop=True)

# Use the last context_size rows of real training outputs as warmup
y_init = y_train.values[-context_size:]

y_pred_msa = multiple_steps_ahead(pipeline, U_val, y_init=y_init)
rmse_msa, nmse_msa = compute_metrics(y_val, y_pred_msa)

# --- Print comparison ---
print("\n" + "=" * 60)
print("Validation Results")
print("=" * 60)
print(f"{'':20} {'RMSE':>10} {'NMSE':>10}")
print("-" * 42)
print(f"{'One-Step-Ahead':<20} {rmse_osa:>10.4f} {nmse_osa:>10.4f}")
print(f"{'Multi-Step-Ahead':<20} {rmse_msa:>10.4f} {nmse_msa:>10.4f}")


Validation Results
                           RMSE       NMSE
------------------------------------------
One-Step-Ahead           0.0491     0.0163
Multi-Step-Ahead         0.0508     0.0174
